In [66]:
# Projet SIEM : Suricata, Filebeat, Elasticsearch

## Objectifs

Ce projet vous permettra de :
- Vérifier le bon fonctionnement de la stack SIEM
- Analyser les données indexées par Filebeat (Suricata)
- Détecter des anomalies sans ML
- Détecter des anomalies avec ML (Isolation Forest)

## Modalité
- groupe de 3 à 4
- output : projet git avec ce notebook détaillé et complété

## Architecture SIEM

```
Suricata   →   Filebeat   →   Elasticsearch   →   Kibana
(IDS/HIDS)   (Collecteur)      (Stockage)   (Visualisation/Alerting)
```

## Prérequis

1. Démarrer la stack :
```bash
docker-compose -f docker-compose-siem.yml up -d
```

2. Attendre quelques minutes que Suricata génère des logs et que Filebeat les indexe dans Elasticsearch.

SyntaxError: invalid character '→' (U+2192) (3498685080.py, line 18)

In [ ]:
# Configuration et connexion à Elasticsearch
from elasticsearch import Elasticsearch
from datetime import datetime, timedelta
import json
import subprocess
import os
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.simplefilter("ignore", InsecureRequestWarning)

# Configuration
ES_HOST = "https://localhost:9200"
ES_USER = "elastic"
ES_PASSWORD = "changeme"  # Modifiez selon votre .env

# Connexion
es = Elasticsearch(
    [ES_HOST],
    basic_auth=(ES_USER, ES_PASSWORD),
    verify_certs=False
)

# Vérification de la connexion
health = es.cluster.health()
print(f"✅ Cluster Elasticsearch: {health['status']} ({health['number_of_nodes']} nœuds)")

✅ Cluster Elasticsearch: green (3 nœuds)


/Users/auteqia/Dev/elasticsearch_cookbook/.venv/lib/python3.13/site-packages/elasticsearch/_sync/client/__init__.py:313: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


## 1. Vérification de la stack

In [67]:
# Vérification des services Docker
services = ['es01', 'es02', 'es03', 'kibana', 'suricata', 'filebeat']
running = []

for service in services:
    try:
        result = subprocess.run(
            ['docker', 'ps', '--filter', f'name={service}', '--format', '{{.Names}}'],
            capture_output=True, text=True, timeout=5
        )
        if service in result.stdout:
            running.append(service)
            print(f"✅ {service}")
        else:
            print(f"❌ {service}")
    except:
        print(f"❌ {service}")

if len(running) == len(services):
    print(f"\n✅ Tous les services sont démarrés ({len(running)}/{len(services)})")
else:
    print(f"\n⚠️  Services démarrés: {len(running)}/{len(services)}")

✅ es01
✅ es02
✅ es03
✅ kibana
✅ suricata
✅ filebeat

✅ Tous les services sont démarrés (6/6)


## 2. Vérification de l'injection des données

In [68]:
# Recherche des index Suricata
def get_suricata_index():
    """Retourne le nom de l'index Suricata le plus récent"""
    try:
        indices = es.indices.get(index="suricata-*")
        if indices:
            return sorted(indices.keys())[-1]
    except:
        pass
    return "suricata-*"

index_name = get_suricata_index()

# Comptage des documents
try:
    count = es.count(index=index_name)
    print(f"📊 Index: {index_name}")
    print(f"📈 Nombre de documents: {count['count']:,}")
    
    # Exemple de document
    if count['count'] > 0:
        sample = es.search(index=index_name, size=1, query={"match_all": {}})
        if sample['hits']['hits']:
            doc = sample['hits']['hits'][0]['_source']
            print(f"\n📄 Exemple de document:")
            print(f"   Type: {doc.get('event_type', 'N/A')}")
            print(f"   Timestamp: {doc.get('@timestamp', doc.get('timestamp', 'N/A'))}")
            if 'src_ip' in doc:
                print(f"   Source: {doc.get('src_ip')}:{doc.get('src_port', 'N/A')}")
                print(f"   Destination: {doc.get('dest_ip')}:{doc.get('dest_port', 'N/A')}")
            if 'alert' in doc:
                alert = doc['alert']
                print(f"   Alerte: {alert.get('signature', 'N/A')}")
                print(f"   Sévérité: {alert.get('severity', 'N/A')}")
except Exception as e:
    print(f"❌ Erreur: {e}")

📊 Index: .ds-suricata-2026.03.17-2026.03.17-000001
📈 Nombre de documents: 2,023

📄 Exemple de document:
   Type: stats
   Timestamp: 2026-03-17T10:38:34.745Z


In [69]:
import platform

# Configuration centralisee pour rendre le notebook plus portable.
# Sur une autre machine, il suffit d'ajuster ces variables.
SIM_TARGET_IP = os.getenv("SIM_TARGET_IP", "192.168.139.2")
SIM_HTTP_URL = os.getenv("SIM_HTTP_URL", f"http://{SIM_TARGET_IP}:80")
SIM_MYSQL_PORT = int(os.getenv("SIM_MYSQL_PORT", "3306"))
USE_SYNTHETIC_FALLBACK = os.getenv("USE_SYNTHETIC_FALLBACK", "false").lower() == "true"

print("Configuration de simulation")
print(f"- OS: {platform.system()} {platform.release()}")
print(f"- SIM_TARGET_IP: {SIM_TARGET_IP}")
print(f"- SIM_HTTP_URL: {SIM_HTTP_URL}")
print(f"- SIM_MYSQL_PORT: {SIM_MYSQL_PORT}")
print(f"- USE_SYNTHETIC_FALLBACK: {USE_SYNTHETIC_FALLBACK}")

Configuration de simulation
- OS: Darwin 25.0.0
- SIM_TARGET_IP: 192.168.139.2
- SIM_HTTP_URL: http://192.168.139.2:80
- SIM_MYSQL_PORT: 3306
- USE_SYNTHETIC_FALLBACK: False


## 3. Portabilite et configuration

Les IPs de simulation ne seront pas forcement les memes sur un autre poste ou un autre OS.

Pourquoi:
- l'adresse IP locale depend du reseau de la machine qui execute le notebook
- Docker Desktop (macOS/Windows) ne se comporte pas exactement comme Docker sur Linux pour la capture reseau
- une cible visible sur une machine peut ne pas etre accessible ou observable sur une autre

Bonne pratique:
- centraliser les parametres de simulation dans une seule cellule
- tester d'abord si le trafic reel apparait dans `suricata-*`
- utiliser l'injection synthetique uniquement si la capture reseau de la plateforme ne voit pas les scans

## 4. Simuler des comportements anormaux

Objectif: generer du trafic anormal dans votre environnement de lab uniquement pour verifier la detection.

Note: macOS + Docker Desktop: le trafic genere depuis le notebook peut ne pas etre visible par Suricata en conteneur.
Pour fiabiliser le TP, les simulations ci-dessous utilisent la configuration de la section 3 et peuvent etre completees par le fallback synthetique.

#### Simulation de scans suspects (en ligne de commande ou en python)

Important: ne scannez pas l'IP de votre propre machine (ni `127.0.0.1`).
Utilisez une cible distante visible par Suricata sur le meme reseau (ex: `192.168.139.3`).

In [ ]:
import subprocess

def _run_python_in_suricata(script):
    """Execute du Python dans le conteneur suricata."""
    result = subprocess.run(
        ["docker", "exec", "suricata", "python3", "-c", script],
        capture_output=True,
        text=True,
        timeout=300,
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())

def simulate_port_scan(target_ip=None, max_port=2000, timeout=0.02):
    """Genere un scan TCP depuis le conteneur suricata"""
    target_ip = target_ip or SIM_TARGET_IP
    print(f"[scan] Lancement depuis le conteneur suricata vers {target_ip} (1-{max_port})")
    script = f"""
import socket
opened = 0
for port in range(1, {max_port} + 1):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout({timeout})
        s.connect((\"{target_ip}\", port))
        s.close()
        opened += 1
    except Exception:
        pass
print(f\"[scan/container] Termine. Ports ouverts detectes: {{opened}}/{max_port}\")
"""
    _run_python_in_suricata(script)

# Exemple d'execution
simulate_port_scan()

[scan] Lancement depuis le conteneur suricata vers 192.168.139.2 (1-2000)
[scan/container] Termine. Ports ouverts detectes: 0/2000


#### Simulation de burst HTTP en ligne de commande (en ligne de commande ou en python) 

In [71]:
import subprocess

def simulate_http_burst(target_url=None, num_requests=300, timeout=1):
    """Genere un burst HTTP depuis le conteneur suricata (lab uniquement)."""
    target_url = target_url or SIM_HTTP_URL
    print(f"[http] Lancement depuis suricata vers {target_url} ({num_requests} requetes)")

    # Utilise curl present dans le conteneur (requests n'est pas installe)
    shell_cmd = (
        f"ok=0; ko=0; "
        f"for i in $(seq 1 {num_requests}); do "
        f"curl -m {timeout} -s -o /dev/null '{target_url}' && ok=$((ok+1)) || ko=$((ko+1)); "
        f"done; "
        f"echo \"[http/container] Termine. OK=$ok, erreurs=$ko\""
    )

    result = subprocess.run(
        ["docker", "exec", "suricata", "bash", "-lc", shell_cmd],
        capture_output=True,
        text=True,
        timeout=300,
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())

# Exemple d'execution
simulate_http_burst()

[http] Lancement depuis suricata vers http://192.168.139.2:80 (300 requetes)
[http/container] Termine. OK=0, erreurs=300


#### Simulation de tentatives de connexion repetees vers MySQL

In [72]:
import subprocess

def simulate_mysql_connections(target_ip=None, target_port=None, num_attempts=50, timeout=0.5):
    """Genere des tentatives TCP vers 3306 depuis le conteneur suricata."""
    target_ip = target_ip or SIM_TARGET_IP
    target_port = target_port or SIM_MYSQL_PORT
    print(f"[mysql] Lancement depuis suricata vers {target_ip}:{target_port} ({num_attempts} essais)")
    script = f"""
import socket
success = 0
failed = 0
for _ in range({num_attempts}):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout({timeout})
        s.connect((\"{target_ip}\", {target_port}))
        s.close()
        success += 1
    except Exception:
        failed += 1
print(f\"[mysql/container] Termine. Succes={{success}}, echecs={{failed}}\")
"""
    result = subprocess.run(
        ["docker", "exec", "suricata", "python3", "-c", script],
        capture_output=True,
        text=True,
        timeout=300,
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())

# Exemple d'execution
simulate_mysql_connections()

[mysql] Lancement depuis suricata vers 192.168.139.2:3306 (50 essais)
[mysql/container] Termine. Succes=0, echecs=50


In [73]:
from datetime import datetime, timezone

def inject_synthetic_flows(attacker_ip="10.10.10.50", victim_ip="192.168.139.2"):
    """Injecte des flux synthetiques pour valider les regles scan/burst/mysql."""
    now_dt = datetime.now(timezone.utc)
    now = now_dt.isoformat()

    # Ecriture vers le data stream du jour (et non vers un backing index)
    write_target = f"suricata-{now_dt:%Y.%m.%d}"
    injected = 0

    # 1) Scan: beaucoup de ports cibles differents
    for p in range(1000, 1060):
        doc = {
            "@timestamp": now,
            "event_type": "flow",
            "src_ip": attacker_ip,
            "dest_ip": victim_ip,
            "dest_port": p,
            "proto": "TCP",
        }
        es.index(index=write_target, document=doc)
        injected += 1

    # 2) Bruit MySQL
    for _ in range(30):
        doc = {
            "@timestamp": now,
            "event_type": "flow",
            "src_ip": attacker_ip,
            "dest_ip": victim_ip,
            "dest_port": 3306,
            "proto": "TCP",
        }
        es.index(index=write_target, document=doc)
        injected += 1

    # 3) Burst HTTP/flow
    for _ in range(80):
        doc = {
            "@timestamp": now,
            "event_type": "flow",
            "src_ip": "10.10.10.60",
            "dest_ip": victim_ip,
            "dest_port": 80,
            "proto": "TCP",
        }
        es.index(index=write_target, document=doc)
        injected += 1

    es.indices.refresh(index=write_target)
    print(f"[inject] {injected} flux synthetiques injectes dans {write_target}")

# Lancez cette cellule si vos simulations reseau reelles ne sont pas visibles
inject_synthetic_flows()

[inject] 170 flux synthetiques injectes dans suricata-2026.03.17


#### Interpretation

Dans notre environnement macOS + Docker Desktop, une partie du trafic de simulation n'etait pas toujours visible par Suricata en mode conteneur. Pour valider objectivement la logique de detection, nous avons completé les tests réseau par une injection de flux synthetiques dans `suricata-*`.

Ce choix garantit un test de bout en bout de la chaine analytique (indexation, agregations, seuils, scoring ML), sans dependre des limites de capture reseau de la plateforme. Les resultats de detection restent interpretable car les patterns injectes reproduisent volontairement trois comportements anormaux: scan multi-ports, burst de trafic, et tentatives repetees vers le port 3306.

## 5. Detection d'anomalies sans ML

### Approche par regles (avec auto-calibration)

Dans ce lab, les evenements `alert` peuvent etre absents (`rules_loaded = 0`).
La detection s'appuie donc surtout sur les `flow` et ajuste les seuils selon le trafic observe sur la fenetre analysee.

Principe:
- afficher un diagnostic rapide (IPs dominantes, ports cibles, pics par minute)
- definir des seuils minimum + ajustement automatique
- detecter 3 types: scan (multiples ports), burst de flux, tentatives MySQL

In [74]:
from datetime import datetime, timedelta, timezone

def detect_anomalies_rules(
    index=index_name,
    lookback_minutes=60,
    min_flows=5,
    scan_ports_floor=8,
    burst_floor_per_minute=12,
    mysql_floor=1,
):
    """Detection sans ML adaptee au trafic reel du lab (principalement flow)."""
    # Fenetre temporelle analysee
    gte_time = (datetime.now(timezone.utc) - timedelta(minutes=lookback_minutes)).isoformat()

    print("\n=== Detection sans ML (regles) ===")
    print(f"Fenetre d'analyse: {lookback_minutes} minutes")

    # 1) Profil global du trafic flow par IP source
    profile_query = {
        "size": 0,
        "query": {
            "bool": {
                "filter": [
                    {"term": {"event_type": "flow"}},
                    {"range": {"@timestamp": {"gte": gte_time}}}
                ]
            }
        },
        "aggs": {
            "by_src": {
                "terms": {"field": "src_ip", "size": 50},
                "aggs": {
                    "unique_dest_ports": {"cardinality": {"field": "dest_port"}},
                    "unique_dest_ips": {"cardinality": {"field": "dest_ip"}},
                    "per_minute": {
                        "date_histogram": {
                            "field": "@timestamp",
                            "fixed_interval": "1m"
                        }
                    }
                }
            }
        }
    }

    profile_res = es.search(index=index, body=profile_query)
    buckets = profile_res["aggregations"]["by_src"]["buckets"]

    if not buckets:
        print("Aucun evenement flow dans la fenetre. Lance d'abord une simulation.")
        return {"scan_suspect": [], "http_burst": [], "mysql_noise": []}

    # 2) Auto-calibration des seuils selon le trafic observe
    max_ports_seen = 0
    max_per_min_seen = 0
    for b in buckets:
        max_ports_seen = max(max_ports_seen, b["unique_dest_ports"]["value"])
        for t in b["per_minute"]["buckets"]:
            max_per_min_seen = max(max_per_min_seen, t["doc_count"])

    # Les floors evitent des seuils trop faibles en cas de trafic calme
    scan_threshold = max(scan_ports_floor, int(max_ports_seen * 0.5))
    burst_threshold = max(burst_floor_per_minute, int(max_per_min_seen * 0.6))

    print("\nDiagnostic flow (top 5 IP source):")
    for b in buckets[:5]:
        ip = b["key"]
        total = b["doc_count"]
        ports = b["unique_dest_ports"]["value"]
        dests = b["unique_dest_ips"]["value"]
        local_peak = 0
        for t in b["per_minute"]["buckets"]:
            local_peak = max(local_peak, t["doc_count"])
        print(f"- {ip}: flows={total}, ports_uniques={ports}, dest_ips={dests}, pic_minute={local_peak}")

    print("\nSeuils utilises:")
    print(f"- scan_threshold (ports uniques): {scan_threshold}")
    print(f"- burst_threshold (flows/min): {burst_threshold}")
    print(f"- mysql_threshold (events vers 3306): {mysql_floor}")

    # 3) Regle scan: source avec beaucoup de ports cibles differents
    scan_hits = []
    for b in buckets:
        total = b["doc_count"]
        ports = b["unique_dest_ports"]["value"]
        dests = b["unique_dest_ips"]["value"]
        if total >= min_flows and ports >= scan_threshold:
            scan_hits.append((b["key"], total, ports, dests))

    # 4) Regle burst: pic de volume sur une fenetre minute
    burst_hits = []
    for b in buckets:
        ip = b["key"]
        for t in b["per_minute"]["buckets"]:
            if t["doc_count"] >= burst_threshold:
                burst_hits.append((ip, t["key_as_string"], t["doc_count"]))

    # 5) Regle mysql: tentatives vers le port 3306
    mysql_query = {
        "size": 0,
        "query": {
            "bool": {
                "filter": [
                    {"term": {"dest_port": 3306}},
                    {"range": {"@timestamp": {"gte": gte_time}}}
                ]
            }
        },
        "aggs": {
            "by_src": {
                "terms": {"field": "src_ip", "size": 50}
            }
        }
    }

    mysql_res = es.search(index=index, body=mysql_query)
    mysql_hits = []
    for b in mysql_res["aggregations"]["by_src"]["buckets"]:
        if b["doc_count"] >= mysql_floor:
            mysql_hits.append((b["key"], b["doc_count"]))

    print("\n[1] Scan suspect (flow)")
    if scan_hits:
        for ip, total, ports, dests in scan_hits:
            print(f"- {ip}: {total} flows, {ports} ports cibles, {dests} IPs destination")
    else:
        print("- Aucun cas detecte")

    print("\n[2] Burst de trafic (flow/min)")
    if burst_hits:
        for ip, minute, count in burst_hits:
            print(f"- {ip}: {count} flows a {minute}")
    else:
        print("- Aucun cas detecte")

    print("\n[3] Bruit MySQL")
    if mysql_hits:
        for ip, count in mysql_hits:
            print(f"- {ip}: {count} tentatives vers le port 3306")
    else:
        print("- Aucun cas detecte")

    return {
        "scan_suspect": scan_hits,
        "http_burst": burst_hits,
        "mysql_noise": mysql_hits,
        "thresholds": {
            "scan_ports": scan_threshold,
            "burst_per_min": burst_threshold,
            "mysql": mysql_floor,
        },
    }

rule_results = detect_anomalies_rules()


=== Detection sans ML (regles) ===
Fenetre d'analyse: 60 minutes

Diagnostic flow (top 5 IP source):
- 10.10.10.50: flows=180, ports_uniques=61, dest_ips=1, pic_minute=90
- 10.10.10.60: flows=160, ports_uniques=1, dest_ips=1, pic_minute=80
- 192.168.139.3: flows=80, ports_uniques=3, dest_ips=2, pic_minute=3
- 192.168.139.2: flows=49, ports_uniques=2, dest_ips=3, pic_minute=9
- fd07:b51a:cc66:0000:a617:db5e:0ab7:e9f1: flows=3, ports_uniques=1, dest_ips=1, pic_minute=1

Seuils utilises:
- scan_threshold (ports uniques): 30
- burst_threshold (flows/min): 54
- mysql_threshold (events vers 3306): 1

[1] Scan suspect (flow)
- 10.10.10.50: 180 flows, 61 ports cibles, 1 IPs destination

[2] Burst de trafic (flow/min)
- 10.10.10.50: 90 flows a 2026-03-17T10:29:00.000Z
- 10.10.10.50: 90 flows a 2026-03-17T10:39:00.000Z
- 10.10.10.60: 80 flows a 2026-03-17T10:29:00.000Z
- 10.10.10.60: 80 flows a 2026-03-17T10:39:00.000Z

[3] Bruit MySQL
- 10.10.10.50: 60 tentatives vers le port 3306


## 6. Detection d'anomalies avec ML

Objectif: construire un detecteur non supervise pour identifier des IPs atypiques.

Comment ca fonctionne:
- on agrege les evenements par IP source
- on transforme chaque IP en vecteur de features (`total_events`, `unique_dest_ports`, `unique_signatures`)
- `IsolationForest` apprend la zone de comportement la plus frequente
- les IPs qui s'ecartent de cette zone sont etiquetees comme anomalies (`-1`)

Ici, on utilise `IsolationForest` sur des features agregees par IP source:
- `total_events`
- `unique_dest_ports`
- `unique_signatures`

Le modele retourne `-1` pour une anomalie et `1` pour un comportement considere normal.

In [75]:
import pandas as pd
from sklearn.ensemble import IsolationForest

def train_and_detect_ml(index=index_name, lookback_minutes=60, contamination=0.05):
    """Detection d'anomalies non supervisee par IP source."""
    gte_time = (datetime.now(timezone.utc) - timedelta(minutes=lookback_minutes)).isoformat()

    # On construit des features agregees par IP source
    query = {
        "size": 0,
        "query": {
            "range": {"@timestamp": {"gte": gte_time}}
        },
        "aggs": {
            "by_ip": {
                "terms": {"field": "src_ip", "size": 1000},
                "aggs": {
                    "unique_dest_ports": {"cardinality": {"field": "dest_port"}},
                    "unique_signatures": {"cardinality": {"field": "alert.signature_id"}}
                }
            }
        }
    }

    res = es.search(index=index, body=query)

    rows = []
    for b in res["aggregations"]["by_ip"]["buckets"]:
        rows.append({
            "ip": b["key"],
            "total_events": b["doc_count"],
            "unique_dest_ports": b["unique_dest_ports"]["value"],
            "unique_signatures": b["unique_signatures"]["value"],
        })

    df = pd.DataFrame(rows)
    if df.empty or len(df) < 5:
        print("Pas assez de donnees pour entrainer IsolationForest (minimum recommande: 5 IPs).")
        return None

    features = ["total_events", "unique_dest_ports", "unique_signatures"]

    # Le modele apprend la zone de normalite et isole les points rares
    model = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=42,
    )

    df["anomaly"] = model.fit_predict(df[features])
    # Plus le score est negatif, plus l'IP est atypique
    df["anomaly_score"] = model.decision_function(df[features])

    anomalies = df[df["anomaly"] == -1].sort_values("anomaly_score")

    print(f"\n=== Detection ML (IsolationForest) sur {len(df)} IPs ===")
    print("\nResume des features (distribution):")
    print(df[features].describe().to_string())

    if anomalies.empty:
        print("\nAucune anomalie detectee par le modele.")
    else:
        print(f"\n{len(anomalies)} anomalie(s) detectee(s):")
        print(anomalies[["ip", "total_events", "unique_dest_ports", "unique_signatures", "anomaly_score"]].to_string(index=False))

        print("\nInterpretation rapide:")
        for row in anomalies.itertuples(index=False):
            reasons = []
            if row.total_events > df["total_events"].quantile(0.9):
                reasons.append("volume d'evenements eleve")
            if row.unique_dest_ports > df["unique_dest_ports"].quantile(0.9):
                reasons.append("beaucoup de ports cibles")
            if row.unique_signatures > df["unique_signatures"].quantile(0.9):
                reasons.append("diversite de signatures elevee")
            if not reasons:
                reasons.append("profil global atypique (combinaison des features)")
            print(f"- {row.ip}: {', '.join(reasons)}; score={row.anomaly_score:.4f}")

    return df, anomalies

ml_results = train_and_detect_ml()


=== Detection ML (IsolationForest) sur 10 IPs ===

Resume des features (distribution):
       total_events  unique_dest_ports  unique_signatures
count     10.000000          10.000000               10.0
mean      67.300000           9.400000                0.0
std       82.071852          19.528896                0.0
min        1.000000           0.000000                0.0
25%        2.250000           0.250000                0.0
50%       16.500000           1.500000                0.0
75%      144.500000           2.750000                0.0
max      194.000000          61.000000                0.0

1 anomalie(s) detectee(s):
         ip  total_events  unique_dest_ports  unique_signatures  anomaly_score
10.10.10.50           180                 61                  0      -0.029737

Interpretation rapide:
- 10.10.10.50: beaucoup de ports cibles; score=-0.0297


## Bonus
- améliorer la stack (ex: ajout de Wazuh)
- dashboard Kibana pour voir en live les simulations de comportements anormaux et les détection d'anomalie (timeline des alertes, etc.)
- analyse statistique avancée
- simulations de comportement anormaux avancée
- utilisation du module de détection d'anomalie d'Elasticsearch (https://www.elastic.co/docs/explore-analyze/machine-learning/anomaly-detection)
- collaboration en groupe sur le projet Git (Pull requests, commits, etc.)
- utilisation de docker / docker compose / devcontainer
- etc.